# Indar - GPU server (free Colab GPU)

Runs the AI room redesign (Dreamshaper 8 img2img) on Google's free GPU and gives your backend a public URL.

**Steps**
1. Menu **Runtime -> Change runtime type -> T4 GPU -> Save**
2. Menu **Runtime -> Run all** (first run takes about 3-4 minutes: it downloads the 2 GB model)
3. Copy the `COMFYUI_URL=...` line printed at the bottom into `indar_app_backend/.env` and restart the backend
4. Keep this tab open while you use the app. The URL changes every time you start the notebook.

In [ ]:
# 1. Check that a GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU - set Runtime > Change runtime type > T4 GPU"

In [ ]:
# 2. Install the server and its dependencies
!mkdir -p /content/indar && cd /content/indar &&   wget -q -O server.py https://raw.githubusercontent.com/Abhin611/Interior-AI/main/gpu_server/server.py &&   wget -q -O requirements.txt https://raw.githubusercontent.com/Abhin611/Interior-AI/main/gpu_server/requirements.txt
!pip install -q -r /content/indar/requirements.txt
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
print("installed")

In [ ]:
# 3. Download Dreamshaper 8 and load it on the GPU (about 2 GB, cached for this session)
import sys
sys.path.insert(0, "/content/indar")
import os; os.chdir("/content/indar")
import server
server.get_pipe()
print("model ready")

In [ ]:
# 4. Start the API server in the background
import threading, time, requests, uvicorn
threading.Thread(target=lambda: uvicorn.run(server.app, host="0.0.0.0", port=8188, log_level="warning"), daemon=True).start()
for _ in range(30):
    try:
        print(requests.get("http://127.0.0.1:8188/system_stats").json()); break
    except Exception:
        time.sleep(1)

In [ ]:
# 5. Open a public tunnel and print the line for your backend .env
import subprocess, re, time
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+[.]trycloudflare[.]com", line)
    if m:
        url = m.group(0)
        break
time.sleep(3)
print()
print("=" * 60)
print("Put this line in indar_app_backend/.env, then restart the backend:")
print()
print(f"COMFYUI_URL={url}")
print("=" * 60)


In [ ]:
# 6. Keep the session alive (stop this cell to shut the server down)
import time
while True:
    time.sleep(60)